# Commodity Data Pipeline

Pulls data from Yahoo Finance, FRED, and EIA into a single daily DataFrame.
Each row is one calendar day from `START_DATE` to `END_DATE`, filled with ffill then bfill.

**Active sources:** Yahoo Finance (prices + state indices), FRED (macro), EIA (energy inventories/production)

## 1. Setup & Configuration

In [ ]:
import os
import json
import requests
import pandas as pd
import yfinance as yf
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()

FRED_API_KEY = os.getenv("FRED_API_KEY")
EIA_API_KEY  = os.getenv("EIA_API_KEY")
NOAA_TOKEN   = os.getenv("NOAA_TOKEN")

START_DATE = "2014-12-28"
END_DATE   = datetime.today().strftime("%Y-%m-%d")
daily_index = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

print(f"Date range: {START_DATE} to {END_DATE} ({len(daily_index)} days)")

## 2. Yahoo Finance - Commodity Prices (Output / Target)

In [ ]:
COMMODITY_TICKERS = {
    "Gold":        "GC=F",
    "Silver":      "SI=F",
    "Crude Oil (WTI)":   "CL=F",
    "Crude Oil (Brent)": "BZ=F",
    "Natural Gas": "NG=F",
    "Copper":      "HG=F",
    "Iron Ore":    "BHP",
    "Lithium":     "LIT",
    "Wheat":       "ZW=F",
    "Soybean":     "ZS=F",
}

def fetch_yahoo_prices(ticker_map, start=START_DATE, end=END_DATE):
    """Fetch daily close prices for all commodities."""
    frames = {}
    for name, ticker in ticker_map.items():
        try:
            data = yf.download(ticker, start=start, end=end, progress=False)
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)
            frames[name] = data["Close"].squeeze().rename(name)
        except Exception as e:
            print(f"Failed to fetch {name} ({ticker}): {e}")
    df = pd.DataFrame(frames)
    # Fill NaN with the average of 2 days before and 2 days after
    for col in df.columns:
        mask = df[col].isna()
        if not mask.any():
            continue
        for idx in df.index[mask]:
            pos = df.index.get_loc(idx)
            surrounding = []
            for offset in [-2, -1, 1, 2]:
                p = pos + offset
                if 0 <= p < len(df) and pd.notna(df[col].iloc[p]):
                    surrounding.append(df[col].iloc[p])
            if surrounding:
                df.at[idx, col] = sum(surrounding) / len(surrounding)
    return df

prices = fetch_yahoo_prices(COMMODITY_TICKERS)
print(f"Shape: {prices.shape}")
prices.tail()

In [ ]:
# Quick visualization - normalize each commodity to 100 at its first available date
prices_norm = prices / prices.apply(lambda col: col.dropna().iloc[0]) * 100
prices_norm.plot(figsize=(14, 6), title="Commodity Prices (Normalized to 100 at First Available Date)")

## 3. Yahoo Finance - State Indices

In [ ]:
state_ticker = {
    "VXD(Dow Volatility Index)": "^VXD",
    "VXN(Nasdaq Volatility Index)": "^VXN",
    "GVZ(Gold Volatility Index)": "^GVZ",
    "EVZ(Energy Volatility Index)": "^EVZ",
    "MSCI World Index": "URTH",
    "MSCI All Country Index": "ACWI",
    "Nikkei 225": "^N225",
    "EURO STOXX 50": "^STOXX50E",
    "Shanghai Composite": "000001.SS",
    "Hang Seng Index": "^HSI"
}

state_yf = fetch_yahoo_prices(state_ticker)
print(f"Shape: {state_yf.shape}")
state_yf.tail()

## 4. FRED API - Macro Indicators

In [ ]:
def fetch_fred_series(series_id, api_key=FRED_API_KEY, start=START_DATE, end=END_DATE):
    """Fetch a single FRED series via the REST API."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id":        series_id,
        "api_key":          api_key,
        "file_type":        "json",
        "observation_start": start,
        "observation_end":   end,
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    obs = resp.json()["observations"]
    df = pd.DataFrame(obs)
    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", "value"]].dropna().set_index("date").rename(columns={"value": series_id})

FRED_SERIES = {
    "DGS10":    "10Y Treasury Yield",
    "T10YIE":   "10Y Breakeven Inflation",
    "DTWEXBGS": "Trade-Weighted USD",
    "INDPRO":   "Industrial Production",
    "DFF": "Fed Funds Rate (Daily Effective)",
    "SP500":    "S&P 500 Index",
    "DJIA":     "Dow Jones Industrial Average",
    "NASDAQCOM": "NASDAQ Composite Index",
    "VIXCLS":   "VIX Index (SP500 Volatility)",
    "OVXCLS":   "OVX Index (Oil Volatility)",
    "GDPC1":      "Gross Domestic Product",
    "A191RL1Q225SBEA": "Real Personal Consumption Expenditures",
    "RSAFS": "Real Retail Sales",
    "CPIAUCSL": "Consumer Price Index (All Urban Consumers)",
    "CPILFESL": "Core CPI (Less Food & Energy)",
    "PCEPI": "Personal Consumption Expenditures Price Index",
    "PCEPILFE": "Core PCE Price Index (Less Food & Energy)",
    "PPIACO": "Producer Price Index (All Commodities)",
    "UNRATE": "Unemployment Rate",
    "PAYEMS": "Total Nonfarm Payrolls",
    "ICSA": "Initial Claims (Unemployment)",
    "DEXCHUS": "USD to CNY Exchange Rate",
}

fred_data = pd.concat([fetch_fred_series(s) for s in FRED_SERIES], axis=1)
print(f"Shape: {fred_data.shape}")
fred_data.tail()

## 5. EIA - Energy Inventories & Production

In [ ]:
def eia_fetch(series_id, start_date=START_DATE, end_date=END_DATE):
    url = f"https://api.eia.gov/v2/seriesid/{series_id}"
    params = {"api_key": EIA_API_KEY, "data[0]": "value"}
    if start_date:
        params["start"] = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    if end_date:
        params["end"] = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    r = requests.get(url, params=params)
    df = pd.DataFrame(r.json()["response"]["data"])
    df["period"] = pd.to_datetime(df["period"])
    return df.set_index("period")["value"].sort_index()

EIA_SERIES = {
    "Crude Inventory (US)": "PET.WCRSTUS1.W",
    "Strategic Petroleum Reserve": "PET.WCSSTUS1.W",
    "Natural Gas Stock": "NG.NW2_EPG0_SWO_R48_BCF.W",
    "US Crude Production": "PET.WCRFPUS2.W",
}

eia_frames = {}
for name, sid in EIA_SERIES.items():
    eia_frames[name] = eia_fetch(sid)

eia_data = pd.DataFrame(eia_frames)
print(f"Shape: {eia_data.shape}")
eia_data.tail()

## 6. GDELT: BigQuery

GDELT events are fetched via BigQuery (not a REST API call in this notebook).  
The query is stored here for reference. Results are loaded from CSV.

In [ ]:
GDELT_QUERY = """
WITH scored_events AS (
  SELECT
    GLOBALEVENTID,
    SQLDATE,
    CAST(LEFT(CAST(SQLDATE AS STRING), 4) AS INT64) AS event_year,
    Actor1Name, Actor2Name,
    Actor1Type1Code, Actor2Type1Code,
    Actor1CountryCode, Actor2CountryCode,
    EventCode, EventRootCode, QuadClass,
    ActionGeo_FullName, ActionGeo_CountryCode,
    GoldsteinScale, NumMentions, NumSources, NumArticles, AvgTone,
    SOURCEURL,
    (
      ABS(GoldsteinScale) * 0.4
      + ABS(AvgTone)      * 0.3
      + LOG(1 + NumArticles) * 0.3
    ) AS impact_score
  FROM `gdelt-bq.gdeltv2.events`
  WHERE
    SQLDATE >= 20150219
    AND IsRootEvent = 1
    AND NumArticles >= 5
    AND (ABS(GoldsteinScale) >= 5 OR ABS(AvgTone) >= 5)
    AND EventRootCode IN (
      '03', '04', '05', '06', '07', '08',
      '10', '11', '12', '13', '14',
      '15', '16', '17', '18', '19', '20'
    )
    AND (
      Actor1Type1Code IN ('GOV', 'MIL', 'REB', 'OPP', 'BUS', 'IGO', 'NGO')
      OR Actor1CountryCode IS NOT NULL
    )
    AND COALESCE(Actor1Type1Code, '') NOT IN ('SPT', 'MED', 'STU')
    AND COALESCE(Actor2Type1Code, '') NOT IN ('SPT', 'MED', 'STU')
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY event_year ORDER BY impact_score DESC) AS rank_in_year
  FROM scored_events
)
SELECT
  event_year, rank_in_year, GLOBALEVENTID, SQLDATE,
  Actor1Name, Actor1Type1Code, Actor1CountryCode,
  Actor2Name, Actor2Type1Code, Actor2CountryCode,
  EventCode, EventRootCode, QuadClass,
  ActionGeo_FullName, ActionGeo_CountryCode,
  GoldsteinScale, NumArticles, AvgTone, impact_score, SOURCEURL
FROM ranked
WHERE rank_in_year <= 200
ORDER BY event_year, rank_in_year;
"""
print("GDELT BigQuery SQL stored in GDELT_QUERY variable.")
print("Run this in BigQuery console and export results to CSV.")

## 7. Combine All Sources into a Single Daily DataFrame

In [ ]:
# Prefix columns to avoid name collisions and clarify source
prices_prefixed = prices.rename(columns={c: f"YF_Price_{c}" for c in prices.columns})
state_prefixed  = state_yf.rename(columns={c: f"YF_State_{c}" for c in state_yf.columns})
fred_prefixed   = fred_data.rename(columns={c: f"FRED_{c}" for c in fred_data.columns})
eia_prefixed    = eia_data.rename(columns={c: f"EIA_{c}" for c in eia_data.columns})

# Combine all on daily index
combined = pd.DataFrame(index=daily_index)
combined.index.name = "date"

for df in [prices_prefixed, state_prefixed, fred_prefixed, eia_prefixed]:
    # Ensure datetime index for joining
    df.index = pd.to_datetime(df.index)
    combined = combined.join(df, how="left")

# Fill: first forward-fill, then back-fill
combined = combined.ffill().bfill()

print(f"Combined shape: {combined.shape}")
print(f"Date range: {combined.index.min()} to {combined.index.max()}")
print(f"Remaining NaNs: {combined.isna().sum().sum()}")
combined.tail()

In [ ]:
combined.to_csv("combined_commodity_data.csv")
print(f"Saved combined_commodity_data.csv ({combined.shape[0]} rows x {combined.shape[1]} columns)")

## 8. Data Dictionary

In [ ]:
data_dict = {
    "metadata": {
        "description": "Data dictionary for the combined commodity dataset",
        "date_range": f"{START_DATE} to {END_DATE}",
        "frequency": "Daily (calendar days, gaps filled with ffill then bfill)",
        "index": "date (datetime)"
    },
    "fields": {
        "YF_Price_Gold":                {"description": "Gold futures daily close price",            "ticker": "GC=F",      "source": "Yahoo Finance", "unit": "USD/oz"},
        "YF_Price_Silver":              {"description": "Silver futures daily close price",          "ticker": "SI=F",      "source": "Yahoo Finance", "unit": "USD/oz"},
        "YF_Price_Crude Oil (WTI)":     {"description": "WTI crude oil futures daily close",        "ticker": "CL=F",      "source": "Yahoo Finance", "unit": "USD/barrel"},
        "YF_Price_Crude Oil (Brent)":   {"description": "Brent crude oil futures daily close",      "ticker": "BZ=F",      "source": "Yahoo Finance", "unit": "USD/barrel"},
        "YF_Price_Natural Gas":         {"description": "Natural gas futures daily close",          "ticker": "NG=F",      "source": "Yahoo Finance", "unit": "USD/MMBtu"},
        "YF_Price_Copper":              {"description": "Copper futures daily close price",         "ticker": "HG=F",      "source": "Yahoo Finance", "unit": "USD/lb"},
        "YF_Price_Iron Ore":            {"description": "Iron ore proxy via BHP Group stock",       "ticker": "BHP",       "source": "Yahoo Finance", "unit": "USD"},
        "YF_Price_Lithium":             {"description": "Lithium proxy via Global X Lithium ETF",   "ticker": "LIT",       "source": "Yahoo Finance", "unit": "USD"},
        "YF_Price_Wheat":               {"description": "Wheat futures daily close price",          "ticker": "ZW=F",      "source": "Yahoo Finance", "unit": "USD/bushel"},
        "YF_Price_Soybean":             {"description": "Soybean futures daily close price",        "ticker": "ZS=F",      "source": "Yahoo Finance", "unit": "USD/bushel"},

        "YF_State_VXD(Dow Volatility Index)":     {"description": "CBOE Dow Jones Volatility Index",          "ticker": "^VXD",      "source": "Yahoo Finance", "unit": "index"},
        "YF_State_VXN(Nasdaq Volatility Index)":  {"description": "CBOE NASDAQ Volatility Index",             "ticker": "^VXN",      "source": "Yahoo Finance", "unit": "index"},
        "YF_State_GVZ(Gold Volatility Index)":    {"description": "CBOE Gold ETF Volatility Index",           "ticker": "^GVZ",      "source": "Yahoo Finance", "unit": "index"},
        "YF_State_EVZ(Energy Volatility Index)":  {"description": "CBOE Energy Sector ETF Volatility Index",  "ticker": "^EVZ",      "source": "Yahoo Finance", "unit": "index"},
        "YF_State_MSCI World Index":              {"description": "iShares MSCI World ETF",                    "ticker": "URTH",      "source": "Yahoo Finance", "unit": "USD"},
        "YF_State_MSCI All Country Index":        {"description": "iShares MSCI ACWI ETF",                     "ticker": "ACWI",      "source": "Yahoo Finance", "unit": "USD"},
        "YF_State_Nikkei 225":                    {"description": "Nikkei 225 Stock Average",                  "ticker": "^N225",     "source": "Yahoo Finance", "unit": "JPY"},
        "YF_State_EURO STOXX 50":                 {"description": "EURO STOXX 50 Index",                       "ticker": "^STOXX50E", "source": "Yahoo Finance", "unit": "EUR"},
        "YF_State_Shanghai Composite":            {"description": "Shanghai Stock Exchange Composite Index",   "ticker": "000001.SS", "source": "Yahoo Finance", "unit": "CNY"},
        "YF_State_Hang Seng Index":               {"description": "Hang Seng Index",                           "ticker": "^HSI",      "source": "Yahoo Finance", "unit": "HKD"},

        "FRED_DGS10":           {"description": "10-Year Treasury Constant Maturity Rate",               "series_id": "DGS10",           "source": "FRED", "unit": "percent",          "frequency_raw": "daily"},
        "FRED_T10YIE":          {"description": "10-Year Breakeven Inflation Rate",                      "series_id": "T10YIE",          "source": "FRED", "unit": "percent",          "frequency_raw": "daily"},
        "FRED_DTWEXBGS":        {"description": "Trade Weighted U.S. Dollar Index (Broad, Goods & Services)", "series_id": "DTWEXBGS",   "source": "FRED", "unit": "index (Jan 2006=100)", "frequency_raw": "daily"},
        "FRED_INDPRO":          {"description": "Industrial Production Index",                            "series_id": "INDPRO",          "source": "FRED", "unit": "index (2017=100)", "frequency_raw": "monthly"},
        "FRED_DFF":             {"description": "Federal Funds Effective Rate",                           "series_id": "DFF",             "source": "FRED", "unit": "percent",          "frequency_raw": "daily"},
        "FRED_SP500":           {"description": "S&P 500 Index",                                         "series_id": "SP500",           "source": "FRED", "unit": "index",            "frequency_raw": "daily"},
        "FRED_DJIA":            {"description": "Dow Jones Industrial Average",                          "series_id": "DJIA",            "source": "FRED", "unit": "index",            "frequency_raw": "daily"},
        "FRED_NASDAQCOM":       {"description": "NASDAQ Composite Index",                                "series_id": "NASDAQCOM",       "source": "FRED", "unit": "index",            "frequency_raw": "daily"},
        "FRED_VIXCLS":          {"description": "CBOE Volatility Index (VIX)",                           "series_id": "VIXCLS",          "source": "FRED", "unit": "index",            "frequency_raw": "daily"},
        "FRED_OVXCLS":          {"description": "CBOE Crude Oil ETF Volatility Index (OVX)",             "series_id": "OVXCLS",          "source": "FRED", "unit": "index",            "frequency_raw": "daily"},
        "FRED_GDPC1":           {"description": "Real Gross Domestic Product",                           "series_id": "GDPC1",           "source": "FRED", "unit": "billions of chained 2017 USD", "frequency_raw": "quarterly"},
        "FRED_A191RL1Q225SBEA": {"description": "Real Personal Consumption Expenditures (% change)",     "series_id": "A191RL1Q225SBEA", "source": "FRED", "unit": "percent",          "frequency_raw": "quarterly"},
        "FRED_RSAFS":           {"description": "Advance Real Retail and Food Services Sales",           "series_id": "RSAFS",           "source": "FRED", "unit": "millions of USD", "frequency_raw": "monthly"},
        "FRED_CPIAUCSL":        {"description": "Consumer Price Index for All Urban Consumers",          "series_id": "CPIAUCSL",        "source": "FRED", "unit": "index (1982-84=100)", "frequency_raw": "monthly"},
        "FRED_CPILFESL":        {"description": "CPI Less Food and Energy (Core CPI)",                   "series_id": "CPILFESL",        "source": "FRED", "unit": "index (1982-84=100)", "frequency_raw": "monthly"},
        "FRED_PCEPI":           {"description": "Personal Consumption Expenditures Price Index",         "series_id": "PCEPI",           "source": "FRED", "unit": "index (2017=100)", "frequency_raw": "monthly"},
        "FRED_PCEPILFE":        {"description": "PCE Price Index Less Food and Energy (Core PCE)",       "series_id": "PCEPILFE",        "source": "FRED", "unit": "index (2017=100)", "frequency_raw": "monthly"},
        "FRED_PPIACO":          {"description": "Producer Price Index for All Commodities",              "series_id": "PPIACO",          "source": "FRED", "unit": "index (1982=100)", "frequency_raw": "monthly"},
        "FRED_UNRATE":          {"description": "Unemployment Rate",                                     "series_id": "UNRATE",          "source": "FRED", "unit": "percent",          "frequency_raw": "monthly"},
        "FRED_PAYEMS":          {"description": "Total Nonfarm Payrolls",                                "series_id": "PAYEMS",          "source": "FRED", "unit": "thousands of persons", "frequency_raw": "monthly"},
        "FRED_ICSA":            {"description": "Initial Claims for Unemployment Insurance",             "series_id": "ICSA",            "source": "FRED", "unit": "number",           "frequency_raw": "weekly"},
        "FRED_DEXCHUS":         {"description": "USD to Chinese Yuan Exchange Rate",                     "series_id": "DEXCHUS",         "source": "FRED", "unit": "CNY per USD",      "frequency_raw": "daily"},

        "EIA_Crude Inventory (US)":       {"description": "Weekly U.S. Ending Stocks of Crude Oil",              "series_id": "PET.WCRSTUS1.W",          "source": "EIA", "unit": "thousand barrels",  "frequency_raw": "weekly"},
        "EIA_Strategic Petroleum Reserve": {"description": "Weekly U.S. Strategic Petroleum Reserve Stocks",     "series_id": "PET.WCSSTUS1.W",          "source": "EIA", "unit": "thousand barrels",  "frequency_raw": "weekly"},
        "EIA_Natural Gas Stock":           {"description": "Weekly Lower 48 States Natural Gas Working Storage", "series_id": "NG.NW2_EPG0_SWO_R48_BCF.W", "source": "EIA", "unit": "billion cubic feet", "frequency_raw": "weekly"},
        "EIA_US Crude Production":         {"description": "Weekly U.S. Field Production of Crude Oil",          "series_id": "PET.WCRFPUS2.W",          "source": "EIA", "unit": "thousand barrels/day", "frequency_raw": "weekly"}
    }
}

with open("data_dictionary.json", "w") as f:
    json.dump(data_dict, f, indent=2)

print(f"Data dictionary saved with {len(data_dict['fields'])} fields.")
print(json.dumps(data_dict["metadata"], indent=2))

---
## Inactive Sections (from original notebook)

The sections below are not currently in use. They are preserved as reference for future integration.

### NOAA API - Weather Data

Weather is a critical demand driver for Natural Gas (HDD/CDD) and a supply driver for Agriculture.  
**Free API token:** https://www.ncdc.noaa.gov/cdo-web/token

```python
def fetch_noaa_data(dataset_id="GSOM", location_id="FIPS:US", datatype=None,
                    start=START_DATE, end=END_DATE, token=NOAA_TOKEN, limit=1000):
    url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    headers = {"token": token}
    params = {
        "datasetid":  dataset_id,
        "locationid": location_id,
        "startdate":  start,
        "enddate":    end,
        "limit":      limit,
        "units":      "standard",
    }
    if datatype:
        params["datatypeid"] = datatype
    resp = requests.get(url, headers=headers, params=params)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    return pd.DataFrame(results)

# Example: monthly avg temperature for US (nat gas HDD/CDD proxy)
# noaa_df = fetch_noaa_data(dataset_id="GSOM", datatype="TAVG",
#                           start="2023-01-01", end="2023-12-31")
```

Useful datatypes: TAVG (avg temp), TMAX/TMIN, PRCP (precipitation), HDD/CDD (heating/cooling degree days)

### USDA NASS QuickStats API - Agriculture Supply

USDA provides production, acreage, and yield data for wheat and soybeans.  
**Free API key:** https://quickstats.nass.usda.gov/api

```python
USDA_API_KEY = os.getenv("USDA_API_KEY")

def fetch_usda_nass(commodity, statistic="PRODUCTION", year=2024, api_key=USDA_API_KEY):
    url = "https://quickstats.nass.usda.gov/api/api_GET/"
    params = {
        "key":              api_key,
        "commodity_desc":   commodity.upper(),
        "statisticcat_desc": statistic,
        "year":             year,
        "agg_level_desc":   "NATIONAL",
        "format":           "json",
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json().get("data", [])
    return pd.DataFrame(data)
```

WASDE Reports (PDF-based) are the top signal for wheat. Access at: https://www.usda.gov/oce/commodity/wasde

### Trading Economics API - Industrial Indicators

Trading Economics provides China PMI, construction data, and steel demand metrics.  
Used for Copper and Iron Ore models.  
**API key required:** https://tradingeconomics.com/api/ (paid tiers for full access)

```python
TE_API_KEY = os.getenv("TE_API_KEY")

def fetch_trading_economics(indicator, country="china", api_key=TE_API_KEY):
    url = f"https://api.tradingeconomics.com/historical/country/{country}/indicator/{indicator}"
    params = {"c": api_key, "f": "json"}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    if "DateTime" in df.columns:
        df["DateTime"] = pd.to_datetime(df["DateTime"])
        df = df.set_index("DateTime").sort_index()
    return df
```

### OPEC Press Room - Supply Policy Signals

OPEC communiques and press releases drive crude oil supply expectations.  
**No API** - requires web scraping.

```python
from bs4 import BeautifulSoup

def fetch_opec_press_releases(max_pages=1):
    articles = []
    base_url = "https://www.opec.org/opec_web/en/press_room/"
    headers = {"User-Agent": "Mozilla/5.0 (research bot)"}
    try:
        resp = requests.get(base_url, headers=headers, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for item in soup.select(".pressRelease, .focusItem, article")[:10]:
            title_tag = item.find(["h3", "h4", "a"])
            if title_tag:
                articles.append({"title": title_tag.get_text(strip=True)})
    except Exception as e:
        print(f"OPEC scraping failed: {e}")
    return pd.DataFrame(articles) if articles else pd.DataFrame()
```

Tip: GDELT is a more reliable alternative for OPEC news monitoring.

### IEA - EV/Lithium Demand Data

The IEA tracks EV adoption and battery demand - the primary driver for lithium prices.  
**No public REST API** - data is available via downloads and reports.

- IEA Global EV Data Explorer (free, downloadable CSV): https://www.iea.org/data-and-statistics/data-tools/global-ev-data-explorer
- IEA API requires institutional subscription
- Proxy via Yahoo Finance for lithium price: `LIT` ETF (already included in commodity prices above)

### Data Source Reference Table

| Source | Cost | API Key? | Registration / Docs |
|--------|------|----------|--------------------|
| FRED | Free | Yes (free) | https://fred.stlouisfed.org/docs/api/api_key.html |
| Yahoo Finance | Free | No | pip install yfinance |
| GDELT | Free | No | https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/ |
| EIA | Free | Yes (free) | https://www.eia.gov/opendata/register.php |
| NOAA | Free | Yes (free) | https://www.ncdc.noaa.gov/cdo-web/token |
| USDA NASS | Free | Yes (free) | https://quickstats.nass.usda.gov/api |
| Trading Economics | Paid | Yes (paid) | https://tradingeconomics.com/api/ |
| IEA | Mixed | Institutional | https://www.iea.org/data-and-statistics |
| OPEC | Free | No (scrape) | https://www.opec.org/opec_web/en/press_room/ |